In [1]:
!pip install saliency lime shap scikit-image -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.2/86.2 kB 3.0 MB/s eta 0:00:00


In [2]:
import os
os.makedirs("/kaggle/temp", exist_ok=True)
!cp /kaggle/input/notebooks/fahimratul/prepare-datasheet-for-model/test.h5 /kaggle/temp/test.h5


In [3]:
import os
for root, dirs, files in os.walk("/kaggle/input/datasets"):
    if "labels" in dirs:
        print(root)
        break

/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier1



"""
10_xai_metrics.py
------------------
Numerical comparison of 6 XAI methods (Grad-CAM, LIME, SHAP, Integrated Gradients, Blur-IG, XRAI)
using IoU, Faithfulness, and Runtime.

Why xbd_root is needed: test.h5 only contains cropped patches, not the original 
building polygons. Ground-truth masks are required to measure IoU, so the original 
JSON is found using the uid and a mask is created from the polygon at that moment—using 
the exact same box used during patch creation (05_prepare_test.py) to ensure the mask 
and patch are perfectly aligned.

Definitions of the three metrics:
  IoU          — Match (intersection / union) between the most important part of the 
                 heatmap (top X%) versus the actual building mask
  Faithfulness — How much the predicted class confidence drops when the important part 
                 is removed (covered with the mean color); (larger value = more faithful explanation)
  Runtime      — Time taken in seconds by each method for a single patch


In [4]:
%%writefile /kaggle/working/10_xai_metrics.py
import argparse
import io
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
from PIL import Image, ImageDraw
from shapely import wkt as shapely_wkt
from torchvision import models

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
CLASSES = ["no-damage", "minor-damage", "major-damage", "destroyed"]

# Must match 05_prepare_test.py precisely, otherwise the mask and patch will not align
CONTEXT_PAD = 10
XAI_METHODS = ["Grad-CAM", "LIME", "SHAP", "Integrated Gradients", "Blur-IG", "XRAI"]


# ==================================================== MODEL (Exact match with training)

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(cout)

    def forward(self, x):
        return F.max_pool2d(F.relu(self.bn(self.conv(x))), 2)


class SimpleCNN(nn.Module):
    def __init__(self, num_classes=4, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32), ConvBlock(32, 64), ConvBlock(64, 128),
            ConvBlock(128, 256), ConvBlock(256, 512),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(dropout),
            nn.Linear(512, 256), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.gap(self.features(x)))


def build_resnet(num_classes=4):
    m = models.resnet50(weights=None)
    m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, num_classes))
    return m


def load_model(model_type, ckpt_path, device):
    model = (SimpleCNN() if model_type == "cnn" else build_resnet()).to(device)
    ck = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ck["model"])
    model.eval()
    target_layer = model.features[-1].conv if model_type == "cnn" else model.layer4[-1]
    return model, target_layer


def get_mean_std(model_type):
    if model_type == "resnet":
        return IMAGENET_MEAN, IMAGENET_STD
    return np.array([0.5, 0.5, 0.5], np.float32), np.array([0.5, 0.5, 0.5], np.float32)


def normalize_for(model_type, arr01):
    mean, std = get_mean_std(model_type)
    arr = (arr01 - mean) / std
    return torch.from_numpy(np.ascontiguousarray(arr.transpose(2, 0, 1))).float()


def norm01(x):
    x = np.asarray(x, dtype=np.float32)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)


# ==================================================== 1. Grad-CAM

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.act = None
        self.grad = None
        target_layer.register_forward_hook(self._fwd)
        target_layer.register_full_backward_hook(self._bwd)

    def _fwd(self, m, i, o):
        self.act = o.detach()

    def _bwd(self, m, gi, go):
        self.grad = go[0].detach()

    def __call__(self, x, class_idx):
        self.model.zero_grad()
        logits = self.model(x)
        logits[0, class_idx].backward()
        w = self.grad.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.act).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return norm01(cam[0, 0].cpu().numpy())


# ==================================================== 2. LIME

def make_lime_explainer():
    from lime import lime_image
    return lime_image.LimeImageExplainer()


def lime_explain(explainer, model, model_type, img01, device, pred_idx, num_samples=500):
    def predict_fn(imgs):
        batch = torch.stack(
            [normalize_for(model_type, im.astype(np.float32)) for im in imgs]).to(device)
        with torch.no_grad():
            return torch.softmax(model(batch), dim=1).cpu().numpy()

    exp = explainer.explain_instance(img01.astype(np.float64), predict_fn,
                                     top_labels=4, hide_color=0, num_samples=num_samples)
    # For use as a heatmap: fill each superpixel with its weight
    from skimage.segmentation import slic
    segments = exp.segments
    weight_map = dict(exp.local_exp[pred_idx])
    heat = np.zeros(segments.shape, dtype=np.float32)
    for seg_id, w in weight_map.items():
        heat[segments == seg_id] = w
    return norm01(heat)


# ==================================================== 3. SHAP

def shap_explain(explainer, x_tensor):
    sv = explainer.shap_values(x_tensor)
    arr = sv[0][0] if isinstance(sv, list) else sv[0]
    if arr.ndim == 4:
        arr = arr[..., 0]
    heat = np.abs(arr).sum(axis=0) if arr.ndim == 3 else np.abs(arr)
    return norm01(heat)


# ==================================================== 4-6. saliency library

def make_call_model_function(model, model_type, device):
    import saliency.core as saliency_core
    mean, std = get_mean_std(model_type)
    mean_t = torch.tensor(mean).view(1, 3, 1, 1).to(device)
    std_t = torch.tensor(std).view(1, 3, 1, 1).to(device)

    def call_model_function(images, call_model_args=None, expected_keys=None):
        x = torch.from_numpy(images.transpose(0, 3, 1, 2)).float().to(device)
        x.requires_grad_(True)
        x_norm = (x - mean_t) / std_t
        out = torch.softmax(model(x_norm), dim=1)
        target = call_model_args["class_idx"]
        selected = out[:, target]
        grads = torch.autograd.grad(selected, x, grad_outputs=torch.ones_like(selected))[0]
        grads = grads.permute(0, 2, 3, 1).detach().cpu().numpy()
        return {saliency_core.base.INPUT_OUTPUT_GRADIENTS: grads}

    return call_model_function


def saliency_explain(method, img01, call_fn, pred_idx, ig_steps=20, blur_steps=40):
    import saliency.core as saliency_core
    args = {"class_idx": pred_idx}
    baseline = np.zeros_like(img01)

    if method == "IG":
        obj = saliency_core.IntegratedGradients()
        attr = obj.GetMask(img01, call_fn, args, x_steps=ig_steps, x_baseline=baseline)
        heat = np.abs(attr).sum(axis=2)
    elif method == "BlurIG":
        obj = saliency_core.BlurIG()
        attr = obj.GetMask(img01, call_fn, args, steps=blur_steps)
        heat = np.abs(attr).sum(axis=2)
    elif method == "XRAI":
        obj = saliency_core.XRAI()
        params = saliency_core.XRAIParameters()
        params.algorithm = "fast"
        heat = obj.GetMask(img01, call_fn, args, extra_parameters=params)
    else:
        raise ValueError(method)
    return norm01(heat)


# ==================================================== Ground-truth mask (from uid)

def crop_box(minx, miny, maxx, maxy, img_w, img_h, pad=CONTEXT_PAD):
    """Matched precisely with the box calculation in 05_prepare_test.py — otherwise
    the mask and patch will not align."""
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    half = max(maxx - minx, maxy - miny) / 2 + pad
    left = max(0, int(cx - half))
    top = max(0, int(cy - half))
    right = min(img_w, int(cx + half))
    bottom = min(img_h, int(cy + half))
    return left, top, right, bottom


def find_file(base, xbd_root, splits, subdir, suffix):
    for sp in splits:
        p = Path(xbd_root) / sp / subdir / f"{base}{suffix}"
        if p.exists():
            return p
    return None


def build_ground_truth_mask(uid, xbd_root, splits, patch_size=128):
    """
    Finds the original JSON using uid (e.g., 'hurricane-florence_00000159_0007'),
    rasterizes that building's polygon, crops out the exact same region using the
    box from patch creation time, resizes it to patch_size, and returns the binary mask.
    Returns None if it fails.
    """
    try:
        base, poly_idx_str = uid.rsplit("_", 1)
        poly_idx = int(poly_idx_str)
    except Exception:
        return None

    jp = find_file(base, xbd_root, splits, "labels", "_post_disaster.json")
    if jp is None:
        return None
    with open(jp) as f:
        meta = json.load(f)
    feats = meta.get("features", {}).get("xy", [])
    if poly_idx >= len(feats):
        return None

    try:
        poly = shapely_wkt.loads(feats[poly_idx]["wkt"])
    except Exception:
        return None

    img_p = find_file(base, xbd_root, splits, "images", "_post_disaster.png")
    if img_p is None:
        return None
    with Image.open(img_p) as im:
        W, H = im.size

    minx, miny, maxx, maxy = poly.bounds
    box = crop_box(minx, miny, maxx, maxy, W, H)
    left, top, right, bottom = box
    if right - left < 4 or bottom - top < 4:
        return None

    # Draw mask at full image size, then crop using the same box (to match coordinates)
    full_mask = Image.new("L", (W, H), 0)
    draw = ImageDraw.Draw(full_mask)
    geoms = poly.geoms if poly.geom_type == "MultiPolygon" else [poly]
    for g in geoms:
        coords = list(g.exterior.coords)
        draw.polygon(coords, fill=255)

    mask_crop = full_mask.crop(box).resize((patch_size, patch_size), Image.NEAREST)
    return np.asarray(mask_crop) > 127


# ==================================================== Metric calculations

def important_region(heat, frac):
    """Converts the top `frac` part of the highest values in the heatmap into a binary mask."""
    thresh = np.percentile(heat, 100 * (1 - frac))
    return heat >= thresh


def compute_iou(heat, gt_mask, frac):
    pred_mask = important_region(heat, frac)
    inter = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    if union == 0:
        return np.nan
    return inter / union


def compute_faithfulness(model, model_type, img01, heat, pred_idx, device, frac):
    important = important_region(heat, frac)
    mean_color = img01.reshape(-1, 3).mean(axis=0)
    perturbed = img01.copy()
    perturbed[important] = mean_color

    x_orig = normalize_for(model_type, img01).unsqueeze(0).to(device)
    x_pert = normalize_for(model_type, perturbed).unsqueeze(0).to(device)
    with torch.no_grad():
        p_orig = torch.softmax(model(x_orig), dim=1)[0, pred_idx].item()
        p_pert = torch.softmax(model(x_pert), dim=1)[0, pred_idx].item()
    return p_orig - p_pert


# ==================================================== Data helpers

def load_labels_uids(h5_path):
    with h5py.File(h5_path, "r") as f:
        g = f["test"]
        n = g["label"].shape[0]
        labels = g["label"][:].astype(int)
        uids = [u.decode() if isinstance(u, bytes) else u for u in g["uid"][:]]
    return n, labels, uids


def get_patch(h5_path, idx):
    with h5py.File(h5_path, "r") as f:
        png = f["test"]["png"][idx].tobytes()
    return np.asarray(Image.open(io.BytesIO(png)).convert("RGB"), dtype=np.float32) / 255.0


# ==================================================== main

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--h5", default="/kaggle/temp/test.h5")
    ap.add_argument("--xbd_root", required=True)
    ap.add_argument("--splits", nargs="+", default=["test", "hold"])
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--model", choices=["cnn", "resnet"], required=True)
    ap.add_argument("--out_dir", default="/kaggle/working/xai_metrics")
    ap.add_argument("--n_samples", type=int, default=20)
    ap.add_argument("--important_frac", type=float, default=0.20,
                    help="Fraction of heatmap top values considered 'important'")
    ap.add_argument("--lime_samples", type=int, default=500)
    ap.add_argument("--ig_steps", type=int, default=20)
    ap.add_argument("--blur_steps", type=int, default=40)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Device: {device} | model: {args.model}")

    model, target_layer = load_model(args.model, args.ckpt, device)
    gradcam = GradCAM(model, target_layer)
    lime_explainer = make_lime_explainer()
    call_fn = make_call_model_function(model, args.model, device)

    n_total, labels, uids = load_labels_uids(args.h5)
    rng = np.random.RandomState(args.seed)
    order = rng.permutation(n_total)

    print("Preparing SHAP explainer...")
    import shap
    bg_idx = rng.choice(n_total, 20, replace=False)
    background = torch.stack(
        [normalize_for(args.model, get_patch(args.h5, i)) for i in bg_idx]).to(device)
    shap_explainer = shap.GradientExplainer(model, background)

    rows = []            # Raw results for each (sample, method)
    collected = 0
    skipped_no_mask = 0

    for idx in order:
        if collected >= args.n_samples:
            break

        uid = uids[idx]
        gt_mask = build_ground_truth_mask(uid, args.xbd_root, args.splits)
        if gt_mask is None:
            skipped_no_mask += 1
            continue

        img01 = get_patch(args.h5, idx)
        true_idx = int(labels[idx])
        x = normalize_for(args.model, img01).unsqueeze(0).to(device)
        with torch.no_grad():
            probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
        pred_idx = int(probs.argmax())

        # ---------- 6 XAI methods: heatmap + runtime ----------
        heats = {}
        times = {}

        t0 = time.perf_counter()
        try:
            heats["Grad-CAM"] = gradcam(x.clone(), pred_idx)
        except Exception as e:
            print(f"  Grad-CAM fail ({uid}): {e}"); heats["Grad-CAM"] = None
        times["Grad-CAM"] = time.perf_counter() - t0

        t0 = time.perf_counter()
        try:
            heats["LIME"] = lime_explain(lime_explainer, model, args.model, img01,
                                         device, pred_idx, args.lime_samples)
        except Exception as e:
            print(f"  LIME fail ({uid}): {e}"); heats["LIME"] = None
        times["LIME"] = time.perf_counter() - t0

        t0 = time.perf_counter()
        try:
            heats["SHAP"] = shap_explain(shap_explainer, x)
        except Exception as e:
            print(f"  SHAP fail ({uid}): {e}"); heats["SHAP"] = None
        times["SHAP"] = time.perf_counter() - t0

        for label, meth in [("Integrated Gradients", "IG"), ("Blur-IG", "BlurIG"),
                            ("XRAI", "XRAI")]:
            t0 = time.perf_counter()
            try:
                heats[label] = saliency_explain(meth, img01, call_fn, pred_idx,
                                                args.ig_steps, args.blur_steps)
            except Exception as e:
                print(f"  {label} fail ({uid}): {e}"); heats[label] = None
            times[label] = time.perf_counter() - t0

        # ---------- Metric calculations ----------
        for method in XAI_METHODS:
            heat = heats.get(method)
            if heat is None:
                continue
            iou = compute_iou(heat, gt_mask, args.important_frac)
            faith = compute_faithfulness(model, args.model, img01, heat,
                                         pred_idx, device, args.important_frac)
            rows.append({
                "uid": uid, "true_class": CLASSES[true_idx],
                "pred_class": CLASSES[pred_idx], "method": method,
                "iou": iou, "faithfulness": faith, "runtime_sec": times[method],
            })

        collected += 1
        print(f"  [{collected}/{args.n_samples}] {uid} done")

    if skipped_no_mask:
        print(f"\n({skipped_no_mask} samples skipped — original JSON/polygon not found)")

    df = pd.DataFrame(rows)
    df.to_csv(out_dir / f"raw_metrics_{args.model}.csv", index=False)

    summary = df.groupby("method").agg(
        mean_IoU=("iou", "mean"),
        mean_Faithfulness=("faithfulness", "mean"),
        mean_Runtime_sec=("runtime_sec", "mean"),
        n=("iou", "count"),
    ).reindex(XAI_METHODS)

    summary.to_csv(out_dir / f"summary_{args.model}.csv")

    print("\n" + "=" * 66)
    print(f"{args.model.upper()} — Numerical comparison of XAI methods "
          f"(Average over {collected} samples)")
    print("=" * 66)
    print(summary.round(4).to_string())
    print(f"\nSaved -> {out_dir}")


if __name__ == "__main__":
    main()

Writing /kaggle/working/10_xai_metrics.py


In [5]:
!python /kaggle/working/10_xai_metrics.py \
    --h5 /kaggle/temp/test.h5 \
    --xbd_root /kaggle/input/datasets/qianlanzz/xbd-dataset/xbd \
    --splits test hold \
    --ckpt /kaggle/input/notebooks/fahimratul/train-of-cnn-resnet50/runs/best_cnn.pt \
    --model cnn \
    --out_dir /kaggle/working/xai_metrics_cnn --n_samples 3

Device: cuda | model: cnn
Preparing SHAP explainer...
100%|████████████████████████████████████████| 500/500 [00:00<00:00, 690.95it/s]
  [1/3] hurricane-harvey_00000179_0016 done
100%|████████████████████████████████████████| 500/500 [00:00<00:00, 719.69it/s]
  [2/3] palu-tsunami_00000113_0117 done
100%|████████████████████████████████████████| 500/500 [00:00<00:00, 710.05it/s]
  [3/3] mexico-earthquake_00000162_0328 done

CNN — Numerical comparison of XAI methods (Average over 3 samples)
                      mean_IoU  mean_Faithfulness  mean_Runtime_sec  n
method                                                                
Grad-CAM                0.2516             0.2035            0.1048  3
LIME                    0.2707             0.1305            0.9018  3
SHAP                    0.1754             0.5608            1.3925  3
Integrated Gradients    0.1027             0.4947            0.0999  3
Blur-IG                 0.0599             0.4140            1.1518  3
XRAI     

In [6]:
!python /kaggle/working/10_xai_metrics.py \
    --h5 /kaggle/temp/test.h5 \
    --xbd_root /kaggle/input/datasets/qianlanzz/xbd-dataset/xbd \
    --splits test hold \
    --ckpt /kaggle/input/notebooks/fahimratul/train-of-cnn-resnet50/runs/best_resnet.pt \
    --model resnet \
    --out_dir /kaggle/working/xai_metrics_resnet --n_samples 3

Device: cuda | model: resnet
Preparing SHAP explainer...
100%|████████████████████████████████████████| 500/500 [00:01<00:00, 421.18it/s]
  [1/3] hurricane-harvey_00000179_0016 done
100%|████████████████████████████████████████| 500/500 [00:01<00:00, 491.71it/s]
  [2/3] palu-tsunami_00000113_0117 done
100%|████████████████████████████████████████| 500/500 [00:00<00:00, 505.93it/s]
  [3/3] mexico-earthquake_00000162_0328 done

RESNET — Numerical comparison of XAI methods (Average over 3 samples)
                      mean_IoU  mean_Faithfulness  mean_Runtime_sec  n
method                                                                
Grad-CAM                0.3927             0.1528            0.0548  3
LIME                    0.2077             0.1391            1.2530  3
SHAP                    0.2354             0.7675            2.3782  3
Integrated Gradients    0.2055             0.5977            0.3409  3
Blur-IG                 0.1060             0.6840            1.6097  3
XRA